In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import resend
import os
import asyncio

In [ ]:
load_dotenv(override=True)

In [ ]:
def send_test_email():
    resend.api_key = os.environ["RESEND_API_KEY"]
    try:
        resend.Emails.send({
            "from": "Ed <onboarding@resend.dev>",
            "to": ["moumita.ray.soft8@gmail.com"],
            "subject": "Test email",
            "text": "This is an important test email",
        })
        print(200)
    except Exception as e:
        print("error:", e)

send_test_email()

In [ ]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [ ]:
sales_Agent1 = Agent(
    name="Professional Sales Agent",
    instructions=instructions1,
    model ="gpt-4o-mini"
)

sales_Agent2 = Agent(
    name="Engaging Sales Agent",
    instructions=instructions2,
    model ="gpt-4o-mini"
)

sales_Agent3 = Agent(
    name="Busy Sales Agent",
    instructions=instructions3,
    model ="gpt-4o-mini"
)

In [ ]:
result = Runner.run_streamed(sales_Agent1,input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta,end="",flush=True)


In [ ]:
message="Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_Agent1,message),
        Runner.run()
    )